# From Pandas to Dask

This notebook introduces Pandas for structured data analysis and then shows how Dask scales the same API to multiple files and larger-than-memory datasets.

In [ ]:
import psutil
import pathlib
import numpy as np
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt
%matplotlib inline

print(f"Available RAM: {psutil.virtual_memory().available / (1024 ** 3):.1f} GB")

## Part 1: Pandas Basics with Simulated Data

We simulate a physics experiment: measuring distance and time with Gaussian uncertainty, then compute derived quantities using vectorized operations.

In [ ]:
n_samples = 1_000
distance_samples_m = np.random.normal(loc=100, scale=10, size=n_samples)
time_samples_s = np.random.normal(loc=10, scale=2, size=n_samples)

df = pd.DataFrame({"distance_m": distance_samples_m, "time_s": time_samples_s})
df.head()

In [ ]:
df.plot.scatter("time_s", "distance_m", xlabel="time [s]", ylabel="distance [m]", grid=True)

### Vectorized Operations

Pandas applies operations to entire columns at C speed — no Python loops needed:

In [ ]:
df["speed_m_per_s"] = df.distance_m / df.time_s
df.speed_m_per_s.hist(bins=50)
plt.xlabel("speed [m/s]")
plt.ylabel("count");

## Part 2: Real-World Data — Weather Station

We download meteorological data from a Polish weather archive and process it first with Pandas, then with Dask.

In [ ]:
# Download weather data from https://meteo.gig.eu (CC BY 4.0)
import requests, re

data_dir = pathlib.Path("meteo")
data_dir.mkdir(exist_ok=True)

base = "https://meteo.gig.eu/archiwum/2025/"
html = requests.get(base).text
txt_files = sorted(re.findall(r'href="(\d{8}\.txt)"', html))
print(f"Found {len(txt_files)} weekly files at {base}")

for fname in txt_files:
    dest = data_dir / fname
    if dest.exists():
        continue
    url = base + fname
    print(f"Downloading {fname}")
    resp = requests.get(url)
    resp.raise_for_status()
    dest.write_bytes(resp.content)

print(f"Data ready: {len(list(data_dir.glob('*.txt')))} files in {data_dir}/")

### Data Schema

The weather station records 36 variables: temperature, humidity, wind, pressure, solar radiation, etc.

In [ ]:
columns = [
    "Date", "Time", "TempOut", "TempHi", "TempLow", "HumOut", "DewPt",
    "WindSpeed", "WindDir", "WindRun", "WindSpeedHi", "WindDirHi", "WindChill",
    "HeatIndex", "THWIndex", "THSWIndex", "Bar", "Rain", "RainRate",
    "SolarRad", "SolarEnergy", "SolarRadHi", "UVIndex", "UVDose", "UVIndexHi",
    "HeatDD", "CoolDD", "TempIn", "HumIn", "DewPtIn", "HeatIn", "ET",
    "WindSamp", "WindTx", "ISSRecept", "ArcInt"
]

### Pandas: Single File

In [ ]:
first_file = next(data_dir.glob("*.txt"))
df_single = pd.read_csv(first_file, sep=r'\s+', names=columns, skiprows=3, na_values=["---", "------"])
df_single.head()

In [ ]:
df_single.plot.scatter(x="TempOut", y="HumOut", title="Single file")

### Pandas: All Files (traditional loop + concat)

The sequential approach: load each file, collect DataFrames, concatenate. This requires all data in memory at once.

In [ ]:
%%time
df_list = []
for file_path in data_dir.glob("*.txt"):
    df_tmp = pd.read_csv(file_path, sep=r'\s+', names=columns, skiprows=3, na_values=["---", "------"])
    df_list.append(df_tmp)
df_all = pd.concat(df_list)
print(f"Total rows: {len(df_all):,}")

### Dask: All Files in One Line

Dask reads all files with a glob pattern and creates a lazy DataFrame. No data is loaded until  is called.

In [ ]:
df_dask = dd.read_csv(
    data_dir / "*.txt", sep=r'\s+', names=columns, skiprows=3,
    na_values=["---", "------"], assume_missing=True
)

# Fix integer columns that may have NaN
for col in ["HumIn", "HumOut", "SolarRad", "SolarRadHi"]:
    df_dask[col] = df_dask[col].fillna(-1).astype("int64")

In [ ]:
# This is lazy — just metadata, no data loaded yet
df_dask

### Triggering Computation

 executes the entire task graph and returns a Pandas DataFrame. Use it only when you need the final result.

In [ ]:
# Compute a single statistic (minimal data transfer)
df_dask.TempOut.mean().compute()

In [ ]:
# Materialize full DataFrame for plotting (all data in memory)
df_dask.compute().plot.scatter(x="TempOut", y="HumOut", s=1, title="All files via Dask")

### Saving to HDF5

Binary formats like HDF5 are much faster to read/write than CSV and support compression.

In [ ]:
h5_path = data_dir / "meteo_dask.h5"
df_dask.to_hdf(h5_path, key="df", mode="w")
print(f"Uncompressed: {h5_path.stat().st_size / (1024**2):.1f} MB")

df_dask.to_hdf(h5_path, key="df", mode="w", complevel=5)
print(f"Compressed:   {h5_path.stat().st_size / (1024**2):.1f} MB")

## Exercise 1

**Compute the mean wind speed** using both Pandas () and Dask (). Verify the results match.

In [ ]:
# Exercise 1: compute mean WindSpeed with both Pandas and Dask

pandas_mean = ___  # YOUR CODE: df_all.WindSpeed...
dask_mean = ___    # YOUR CODE: df_dask.WindSpeed...

print(f"Pandas mean wind speed: {pandas_mean:.4f}")
print(f"Dask mean wind speed:   {dask_mean:.4f}")

## Exercise 2

**Scale the physics simulation to 100 million samples.** Regenerate the distance/time DataFrame. What happens to the speed histogram shape? Does it still look Gaussian?

In [ ]:
# Exercise 2: scale to 100M samples

n_samples_large = 100_000_000

# YOUR CODE HERE: create df_large with distance_m and time_s columns
# Then compute speed and plot the histogram with bins=200